In [29]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine
from collections import Counter

df = pd.read_csv('ipl_master_players_final.csv')

# All binary features your engine will reason over
VECTOR_COLUMNS = [
    'is_opener', 'is_finisher', 'is_death_bowler', 'is_economical',
    'is_wicketkeeper', 'is_captain', 'is_power_hitter', 'is_spinner',
    'is_pacer', 'is_all_rounder', 'is_anchor', 'is_clutch',
    'active_recent', 'early_ipl_era', 'one_franchise',
    'played_for_csk', 'played_for_mi', 'played_for_rcb',
    'played_for_kkr', 'played_for_srh', 'played_for_dc',
    'played_for_rr', 'played_for_pbks', 'won_ipl_title',
    'won_orange_cap', 'won_purple_cap', 'is_international_star',
    'is_wicketkeeper', 'is_captain'
]
# Remove duplicates
VECTOR_COLUMNS = list(dict.fromkeys(VECTOR_COLUMNS))

# Natural language rendering — no hardcoded game logic, just display
FEATURE_QUESTIONS = {
    'is_opener':           "Is your player an opener?",
    'is_finisher':         "Is your player known for finishing matches?",
    'is_death_bowler':     "Is your player a death-over specialist bowler?",
    'is_economical':       "Is your player known for being economical (low economy rate)?",
    'is_wicketkeeper':     "Is your player a wicketkeeper?",
    'is_captain':          "Has your player captained an IPL team?",
    'is_power_hitter':     "Is your player a power hitter?",
    'is_spinner':          "Is your player primarily a spinner?",
    'is_pacer':            "Is your player primarily a pace bowler?",
    'is_all_rounder':      "Is your player an all-rounder?",
    'is_anchor':           "Is your player known for anchoring the innings?",
    'is_clutch':           "Is your player known for performing under pressure?",
    'active_recent':       "Is your player active in recent IPL seasons?",
    'early_ipl_era':       "Did your player play in the early IPL era (2007–2012)?",
    'one_franchise':       "Has your player played for only one IPL franchise?",
    'played_for_csk':      "Has your player played for Chennai Super Kings?",
    'played_for_mi':       "Has your player played for Mumbai Indians?",
    'played_for_rcb':      "Has your player played for Royal Challengers Bangalore?",
    'played_for_kkr':      "Has your player played for Kolkata Knight Riders?",
    'played_for_srh':      "Has your player played for Sunrisers Hyderabad?",
    'played_for_dc':       "Has your player played for Delhi Capitals/Daredevils?",
    'played_for_rr':       "Has your player played for Rajasthan Royals?",
    'played_for_pbks':     "Has your player played for Punjab Kings/KXIP?",
    'won_ipl_title':       "Has your player won the IPL title?",
    'won_orange_cap':      "Has your player won the Orange Cap?",
    'won_purple_cap':      "Has your player won the Purple Cap?",
    'is_international_star': "Is your player an international star (played for national team)?",
}

# Build player vectors: True→1.0, False→-1.0
vector_df = df[VECTOR_COLUMNS].copy()
for col in VECTOR_COLUMNS:
    vector_df[col] = vector_df[col].map({True: 1.0, False: 0.0}).fillna(0.0)

player_vectors = {
    df.iloc[idx]['player_name']: vector_df.iloc[idx].values
    for idx in range(len(df))
}

print(f"✅ Vector space ready: {len(player_vectors)} players × {len(VECTOR_COLUMNS)} features")

✅ Vector space ready: 788 players × 27 features


In [28]:
def get_scores(user_vec, user_answered_mask):
    """
    Score = dot product only over features the user has actually answered.
    +1 for correct match, -1 for contradiction, 0 for unanswered.
    """
    scores = []
    for name, p_vec in player_vectors.items():
        score = 0.0
        for idx in range(len(VECTOR_COLUMNS)):
            if user_answered_mask[idx] == 0:
                continue  # Skip unanswered features entirely

            user_ans = user_vec[idx]      # +1 = Yes, -1 = No
            player_val = p_vec[idx]       # 1.0 = True, 0.0 = False

            if user_ans == 1.0 and player_val == 1.0:
                score += 2.0   # Strong match
            elif user_ans == -1.0 and player_val == 0.0:
                score += 1.0   # Correct negative match
            elif user_ans == 1.0 and player_val == 0.0:
                score -= 2.0   # Contradiction — they said Yes, player is False
            elif user_ans == -1.0 and player_val == 1.0:
                score -= 2.0   # Contradiction — they said No, player is True

        scores.append((name, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

def entropy_of_feature(feature_idx, top_candidates):
    """
    Calculate how well this feature splits the top candidates.
    Lower entropy = better split = more information gained.
    We want the feature closest to a 50/50 split.
    """
    values = [player_vectors[name][feature_idx] for name in top_candidates]
    count_true  = sum(1 for v in values if v == 1.0)
    count_false = len(values) - count_true

    if count_true == 0 or count_false == 0:
        return 0.0  # Useless feature — everyone is same

    p = count_true / len(values)
    # Shannon entropy: maximized at p=0.5 (perfect split)
    entropy = -(p * np.log2(p) + (1-p) * np.log2(1-p))
    return entropy

def pick_best_question(user_vec, asked_features):
    scores = get_scores(user_vec, np.ones(len(VECTOR_COLUMNS)))  # ← fixed line
    top_candidates = [name for name, _ in scores[:20]]

    best_feature_idx = None
    best_entropy = -1

    for idx, feature in enumerate(VECTOR_COLUMNS):
        if feature in asked_features:
            continue
        e = entropy_of_feature(idx, top_candidates)
        if e > best_entropy:
            best_entropy = e
            best_feature_idx = idx

    return best_feature_idx

print("✅ Entropy engine ready")

✅ Entropy engine ready


In [30]:
import numpy as np

MATCH_REWARD = 3.0
NEGATIVE_MATCH_REWARD = 1.5
CONTRADICTION_PENALTY = 5.0

def get_scores(user_vec, user_answered_mask):
    """
    Scores players based ONLY on answered features.

    Strongly punishes contradictions.
    """

    scores = []

    for name, p_vec in player_vectors.items():

        score = 0.0
        contradictions = 0

        for idx in range(len(VECTOR_COLUMNS)):

            if user_answered_mask[idx] == 0:
                continue

            user_ans = user_vec[idx]      # +1 yes, -1 no
            player_val = p_vec[idx]       # 1 true, 0 false

            # YES and player has feature
            if user_ans == 1.0 and player_val == 1.0:
                score += MATCH_REWARD

            # NO and player truly lacks feature
            elif user_ans == -1.0 and player_val == 0.0:
                score += NEGATIVE_MATCH_REWARD

            # CONTRADICTIONS
            elif user_ans == 1.0 and player_val == 0.0:
                score -= CONTRADICTION_PENALTY
                contradictions += 1

            elif user_ans == -1.0 and player_val == 1.0:
                score -= CONTRADICTION_PENALTY
                contradictions += 1

        # EXTRA punishment for impossible candidates
        if contradictions >= 3:
            score -= contradictions * 3

        scores.append((name, score))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores


def entropy_of_feature(feature_idx, top_candidates):
    """
    Measures how evenly this feature splits candidates.
    Higher entropy = better question.
    """

    values = [
        player_vectors[name][feature_idx]
        for name in top_candidates
    ]

    count_true = sum(values)
    count_false = len(values) - count_true

    # useless split
    if count_true == 0 or count_false == 0:
        return -1

    p_true = count_true / len(values)
    p_false = count_false / len(values)

    entropy = -(
        p_true * np.log2(p_true) +
        p_false * np.log2(p_false)
    )

    return entropy


def pick_best_question(user_vec, user_answered_mask, asked_features):
    """
    Pick the unanswered feature with maximum entropy
    among current top candidates.
    """

    # ONLY use answered features
    scores = get_scores(user_vec, user_answered_mask)

    # narrower candidate pool
    top_candidates = [
        name for name, score in scores[:30]
    ]

    best_feature_idx = None
    best_entropy = -1

    for idx, feature in enumerate(VECTOR_COLUMNS):

        if feature in asked_features:
            continue

        # skip unanswered/non-boolean junk
        values = [
            player_vectors[name][idx]
            for name in top_candidates
        ]

        unique_vals = set(values)

        if len(unique_vals) < 2:
            continue

        e = entropy_of_feature(idx, top_candidates)

        if e > best_entropy:
            best_entropy = e
            best_feature_idx = idx

    return best_feature_idx


print("✅ Improved entropy engine ready")

✅ Improved entropy engine ready


The Full Game Loop (run this to play)

In [38]:

def play_akinator():
    user_vector = np.zeros(len(VECTOR_COLUMNS))
    user_answered_mask = np.zeros(len(VECTOR_COLUMNS))  # Track what was actually answered
    asked_features = set()

    print("🏏 IPL AKINATOR — Think of any IPL player!\n")
    print("The data set is till 2024. kindly keep that in mind.")

    for step in range(12):
        best_idx = pick_best_question(
    user_vector,
    user_answered_mask,
    asked_features
)
        if best_idx is None:
            break

        feature = VECTOR_COLUMNS[best_idx]
        question = FEATURE_QUESTIONS.get(feature, f"Is '{feature}' true?")
        asked_features.add(feature)

        print(f"Q{step+1}: {question}")
        ans = input("   → (y / n / d): ").strip().lower()

        if ans == 'y':
            user_vector[best_idx] = 1.0
            user_answered_mask[best_idx] = 1  # Mark as answered
        elif ans == 'n':
            user_vector[best_idx] = -1.0
            user_answered_mask[best_idx] = 1  # Mark as answered
        else:
            user_vector[best_idx] = 0.0
            user_answered_mask[best_idx] = 0  # Don't know = don't count it

        top = get_scores(user_vector, user_answered_mask)
        top5 = top[:5]

        print(f"\n   📊 Top candidates:")
        for name, score in top5:
            print(f"      {name:<28} score: {score:+.1f}")

        # Confidence check: clear leader with good gap
        if top5[0][1] > top5[1][1] + 3.0 and step >= 4:
            print(f"\n{'='*50}")
            print(f"🎯 I'm confident! Your player is...")
            print(f"   ✨  {top5[0][0].upper()}  ✨")
            print(f"{'='*50}")
            return
        print()

    top = get_scores(user_vector, user_answered_mask)
    print(f"\n{'='*50}")
    print(f"🏆 Final guess: {top[0][0].upper()}")
    print(f"   Runner-up:   {top[1][0]}")
    print(f"{'='*50}")

play_akinator()

🏏 IPL AKINATOR — Think of any IPL player!

The data set is till 2024. kindly keep that in mind.
Q1: Is your player an all-rounder?
   → (y / n / d): n

   📊 Top candidates:
      R Dravid                     score: +1.5
      W Jaffer                     score: +1.5
      V Kohli                      score: +1.5
      MV Boucher                   score: +1.5
      P Kumar                      score: +1.5

Q2: Is your player primarily a pace bowler?
   → (y / n / d): n

   📊 Top candidates:
      R Dravid                     score: +3.0
      SB Joshi                     score: +3.0
      S Badrinath                  score: +3.0
      K Goel                       score: +3.0
      KC Sangakkara                score: +3.0

Q3: Has your player played for only one IPL franchise?
   → (y / n / d): y

   📊 Top candidates:
      SB Joshi                     score: +6.0
      S Badrinath                  score: +6.0
      K Goel                       score: +6.0
      D Salunkhe               

     player_name  is_spinner  is_all_rounder  played_for_csk  active_recent  early_ipl_era  is_international_star  played_for_mi  played_for_kkr  played_for_rcb
386  Imran Tahir        True           False           False          False          False                   True          False           False           False
